# Story Relay Robot Story Relay Robot

## Overview

This notebook demonstrates how to create an interactive story continuation system using a local Large Language Model (LLM) deployed via LM Studio. You will learn:

- **RESTful API Concepts**: Understanding how to interact with APIs using HTTP requests
- **LLM Fundamentals**: Working with Large Language Models for text generation
- **Prompt Engineering**: Crafting effective prompts to guide AI responses
- **OpenAI-Compatible API**: Using the OpenAI API format to communicate with local models

## Prerequisites

Before starting, ensure:
1. LM Studio is installed and running
2. A language model is loaded in LM Studio
3. The local server is running on `http://127.0.0.1:1234`
4. Follow the root README Quick start to create the shared `.venv`. Activate it with `.venv\Scripts\activate` in a Windows terminal or `source .venv/bin/activate` on Linux/macOS, then enter `Story_Relay_Robot/`, run `jupyter notebook`, and select the standard **Python 3 (ipykernel)** kernel. This notebook's language-model request remains HTTP-only and is not routed through ROCm.



![Story Relay Robot Flow](images/story_relay_flow.png)

## Step 1: Import Required Libraries

First, we need to import the necessary libraries for making HTTP requests and handling JSON data. The `requests` library allows us to send HTTP requests to the LM Studio API, and `json` helps us format and parse the data.


In [ ]:
import requests
import json
from typing import Optional, List

# LM Studio API configuration
LM_STUDIO_URL = "http://127.0.0.1:1234/v1/chat/completions"
API_KEY = "not-needed"  # LM Studio doesn't require an API key for local use

def process_streaming_response(response, show_streaming: bool = True) -> str:
    """
    Helper function to process SSE streaming responses from LM Studio.
    
    Args:
        response: The requests.Response object with stream=True
        show_streaming: If True, display text as it streams in real-time
    
    Returns:
        The complete response text, or empty string if error occurred
    """
    full_response = ""
    last_content = ""  # Track last content to detect repetition
    repetition_count = 0
    
    # Handle non-200 status codes
    if response.status_code != 200:
        if show_streaming:
            print(f"Error: API returned status code {response.status_code}")
            try:
                error_text = response.text
                if error_text:
                    print(f"Response: {error_text}")
            except:
                pass
        return ""
    
    try:
        if show_streaming:
            print("Streaming response (real-time):")
            print("-" * 70)
        
        # Process each line in the SSE stream
        for line in response.iter_lines():
            if line:
                # Decode bytes to string
                line_text = line.decode('utf-8')
                
                # SSE format: lines start with "data: "
                if line_text.startswith('data: '):
                    # Extract JSON data
                    json_str = line_text[6:]  # Remove "data: " prefix
                    
                    # Check for [DONE] marker
                    if json_str.strip() == '[DONE]':
                        if show_streaming:
                            print("\n" + "-" * 70)
                            print("Stream complete!")
                        break
                    
                    try:
                        # Parse JSON chunk
                        chunk = json.loads(json_str)
                        
                        # Check for errors in the chunk
                        if 'error' in chunk:
                            if show_streaming:
                                print(f"\nError: {chunk['error']}")
                            break
                        
                        # Extract content delta
                        if 'choices' in chunk and len(chunk['choices']) > 0:
                            choice = chunk['choices'][0]
                            
                            # Check finish_reason to detect end of generation
                            if 'finish_reason' in choice and choice['finish_reason']:
                                if show_streaming:
                                    print("\n" + "-" * 70)
                                    print(f"Stream complete! (Reason: {choice['finish_reason']})")
                                break
                            
                            delta = choice.get('delta', {})
                            content = delta.get('content', '')
                            
                            if content:
                                # Detect repetition: if same content appears multiple times, stop
                                if content == last_content and len(content) > 10:
                                    repetition_count += 1
                                    if repetition_count > 3:
                                        if show_streaming:
                                            print("\n" + "-" * 70)
                                            print("Warning: Detected repetition, stopping stream.")
                                        break
                                else:
                                    repetition_count = 0
                                    last_content = content
                                
                                # Print immediately if streaming display is enabled
                                if show_streaming:
                                    print(content, end='', flush=True)
                                full_response += content
                                
                    except json.JSONDecodeError as e:
                        # Skip invalid JSON lines
                        if show_streaming:
                            print(f"\nWarning: Skipped invalid JSON line: {str(e)[:50]}")
                        continue
                    except UnicodeDecodeError as e:
                        # Handle encoding errors
                        if show_streaming:
                            print(f"\nWarning: Encoding error: {str(e)[:50]}")
                        continue
        
        if show_streaming:
            print()  # New line after streaming
    
    except requests.exceptions.ChunkedEncodingError:
        if show_streaming:
            print("\nWarning: Connection interrupted during streaming.")
    except Exception as e:
        if show_streaming:
            print(f"\nError processing stream: {str(e)}")
    finally:
        # Ensure response is closed
        try:
            response.close()
        except:
            pass
    
    return full_response.strip()

print("Libraries imported successfully!")
print(f"LM Studio API endpoint: {LM_STUDIO_URL}")


## Step 2: Understanding RESTful API and LM Studio Endpoints

### What is a RESTful API?

REST (Representational State Transfer) is an architectural style for designing web services. Key concepts:

- **HTTP Methods**: GET (retrieve), POST (create/send), PUT (update), DELETE (remove)
- **Endpoints**: URLs that represent resources (e.g., `/v1/chat/completions`)
- **Request/Response**: Client sends a request, server returns a response
- **JSON Format**: Data is exchanged in JSON (JavaScript Object Notation) format

### LM Studio API Endpoints

LM Studio provides several OpenAI-compatible API endpoints. The base URL is `http://127.0.0.1:1234`. Here are the main endpoints:

#### 1. GET `/v1/models`
**Purpose**: List all available models in LM Studio

**Request Example**:
```python
response = requests.get("http://127.0.0.1:1234/v1/models")
models = response.json()
```

**Response Format**:
```json
{
  "data": [
    {
      "id": "model-name",
      "object": "model",
      "created": 1234567890,
      "owned_by": "local"
    }
  ]
}
```

#### 2. POST `/v1/chat/completions`
**Purpose**: Generate chat completions using a conversation format (most commonly used)

**Request Format (Non-Streaming)**:
```json
{
  "model": "local-model",
  "messages": [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hello!"}
  ],
  "temperature": 0.7,
  "max_tokens": 200,
  "stream": false  // or omit for non-streaming
}
```

**Request Format (Streaming with SSE)**:
```json
{
  "model": "local-model",
  "messages": [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hello!"}
  ],
  "temperature": 0.7,
  "max_tokens": 200,
  "stream": true  // Enable Server-Sent Events (SSE) streaming
}
```

**Response Format (Non-Streaming)**:
```json
{
  "id": "chatcmpl-123",
  "object": "chat.completion",
  "created": 1234567890,
  "choices": [{
    "index": 0,
    "message": {
      "role": "assistant",
      "content": "Hello! How can I help you?"
    },
    "finish_reason": "stop"
  }],
  "usage": {
    "prompt_tokens": 10,
    "completion_tokens": 8,
    "total_tokens": 18
  }
}
```

**Response Format (Streaming - SSE)**:
When `"stream": true`, the response uses Server-Sent Events (SSE) format. Each line is a JSON object:
```
data: {"id":"chatcmpl-123","object":"chat.completion.chunk","choices":[{"delta":{"content":"Hello"},"index":0}]}

data: {"id":"chatcmpl-123","object":"chat.completion.chunk","choices":[{"delta":{"content":" world"},"index":0}]}

data: [DONE]
```

**Note**: See Step 3.5 for detailed streaming implementation examples.

**Text and Image Passing**:
- **Text**: Passed directly in the `content` field as a string
- **Images**: For vision models, use base64-encoded images in the message content:
```json
{
  "role": "user",
  "content": [
    {"type": "text", "text": "What's in this image?"},
    {
      "type": "image_url",
      "image_url": {
        "url": "data:image/jpeg;base64,/9j/4AAQSkZJRg..."
      }
    }
  ]
}
```

#### 3. POST `/v1/completions`
**Purpose**: Generate text completions (legacy format, simpler than chat)

**Request Format**:
```json
{
  "model": "local-model",
  "prompt": "Once upon a time",
  "temperature": 0.7,
  "max_tokens": 200,
  "stop": ["\n\n"]
}
```

**Response Format**:
```json
{
  "id": "cmpl-123",
  "object": "text_completion",
  "created": 1234567890,
  "choices": [{
    "text": " there was a magical kingdom...",
    "index": 0,
    "finish_reason": "stop"
  }],
  "usage": {
    "prompt_tokens": 4,
    "completion_tokens": 10,
    "total_tokens": 14
  }
}
```

**Text Passing**: The prompt is passed directly as a string in the `prompt` field.

#### 4. POST `/v1/embeddings`
**Purpose**: Generate vector embeddings for text (useful for semantic search, similarity, etc.)

**Request Format**:
```json
{
  "model": "local-model",
  "input": "The text to embed"
}
```

**Response Format**:
```json
{
  "object": "list",
  "data": [{
    "object": "embedding",
    "embedding": [0.1, 0.2, 0.3, ...],
    "index": 0
  }],
  "model": "local-model",
  "usage": {
    "prompt_tokens": 5,
    "total_tokens": 5
  }
}
```

**Text Passing**: Text is passed as a string or array of strings in the `input` field.

#### 5. POST `/v1/responses`
**Purpose**: Alternative endpoint for streaming responses (less commonly used)

**Request Format**: Similar to `/v1/chat/completions` but may support streaming

**Note**: This endpoint may vary depending on LM Studio version. Check the documentation for your specific version.

### Data Passing Methods

#### Text Data
Text is always passed as strings in JSON format:
- In `messages[].content` for chat completions
- In `prompt` for completions
- In `input` for embeddings

#### Image Data
For vision-capable models, images can be passed using:

1. **Base64 Encoding** (most common):
```python
import base64

with open("image.jpg", "rb") as image_file:
    base64_image = base64.b64encode(image_file.read()).decode('utf-8')
    
message_content = [
    {"type": "text", "text": "Describe this image"},
    {
        "type": "image_url",
        "image_url": {
            "url": f"data:image/jpeg;base64,{base64_image}"
        }
    }
]
```

2. **URL** (if the image is accessible via URL):
```json
{
  "type": "image_url",
  "image_url": {
    "url": "https://example.com/image.jpg"
  }
}
```

### Common Request Headers

```python
headers = {
    "Content-Type": "application/json",
    "Authorization": "Bearer not-needed"  # LM Studio doesn't require auth for local use
}
```


### Example: Testing API Endpoints

Let's try out some of these API endpoints to see how they work in practice.


In [ ]:
# Base URL for LM Studio
BASE_URL = "http://127.0.0.1:1234"

# Example 1: List available models
print("=" * 70)
print("Example 1: Listing Available Models")
print("=" * 70)
try:
    response = requests.get(f"{BASE_URL}/v1/models")
    if response.status_code == 200:
        models_data = response.json()
        print(f"Found {len(models_data.get('data', []))} model(s):")
        for model in models_data.get('data', []):
            print(f"  - Model ID: {model.get('id', 'N/A')}")
            print(f"    Object: {model.get('object', 'N/A')}")
    else:
        print(f"Error: Status code {response.status_code}")
except Exception as e:
    print(f"Error listing models: {str(e)}")
    print("Make sure LM Studio is running and a model is loaded.")

print("\n" + "=" * 70)
print("Example 2: Using /v1/completions (Legacy Format)")
print("=" * 70)

# Example 2: Using completions endpoint
completions_payload = {
    "model": "local-model",
    "prompt": "Once upon a time, in a magical forest",
    "temperature": 0.7,
    "max_tokens": 50
}

try:
    response = requests.post(
        f"{BASE_URL}/v1/completions",
        json=completions_payload,
        headers={"Content-Type": "application/json"},
        timeout=30
    )
    if response.status_code == 200:
        result = response.json()
        if result.get('choices'):
            print(f"Prompt: {completions_payload['prompt']}")
            print(f"Completion: {result['choices'][0].get('text', 'N/A')}")
            print(f"Tokens used: {result.get('usage', {}).get('total_tokens', 'N/A')}")
    else:
        print(f"Error: Status code {response.status_code}")
        print(f"Response: {response.text}")
except Exception as e:
    print(f"Error: {str(e)}")

print("\n" + "=" * 70)
print("Note: The /v1/chat/completions endpoint will be used in the following steps")
print("=" * 70)


## Step 3: Test Connection to LM Studio

Before we start creating stories, let's verify that we can connect to LM Studio. This function will send a simple test message to check if the API is accessible.


## Step 3.5: Understanding Streaming Responses and SSE (Server-Sent Events)

### What is Streaming?

**Streaming** allows you to receive data incrementally as it's generated, rather than waiting for the complete response. This is especially useful for LLMs because:

- **Real-time feedback**: See text appear as it's generated (like ChatGPT)
- **Better user experience**: Users don't have to wait for the entire response
- **Lower perceived latency**: Content appears immediately

### SSE (Server-Sent Events)

**SSE (Server-Sent Events)** is a web standard that enables servers to push data to clients over HTTP. It's simpler than WebSockets and perfect for one-way streaming (server → client).

**How SSE Works**:
1. Client sends a request with `stream=True`
2. Server keeps the connection open
3. Server sends data chunks as `data: {...}` lines
4. Client processes each chunk as it arrives
5. Connection closes when complete

### Does `requests` Library Support Streaming?

**Yes!** The `requests` library supports streaming through the `stream=True` parameter:

```python
response = requests.post(url, json=payload, stream=True)
for line in response.iter_lines():
    # Process each line
```

### Does LM Studio Support Streaming?

**Yes!** LM Studio supports streaming responses when you set `"stream": true` in your request payload. The response format changes to SSE format.

**Non-streaming Request**:
```json
{
  "model": "local-model",
  "messages": [...],
  "stream": false  // or omit this field
}
```

**Streaming Request**:
```json
{
  "model": "local-model",
  "messages": [...],
  "stream": true  // Enable streaming
}
```

**Streaming Response Format**:
Each chunk is a line starting with `data: ` followed by JSON:
```
data: {"id":"chatcmpl-123","object":"chat.completion.chunk","choices":[{"delta":{"content":"Hello"},"index":0}]}

data: {"id":"chatcmpl-123","object":"chat.completion.chunk","choices":[{"delta":{"content":" world"},"index":0}]}

data: [DONE]
```

The `[DONE]` marker indicates the stream is complete.


In [ ]:
def generate_story_streaming(
    story_so_far: str,
    temperature: float = 0.8,
    max_tokens: int = 200
) -> str:
    """
    Generate a story continuation using streaming mode.
    This function demonstrates SSE (Server-Sent Events) streaming with LM Studio.
    
    Args:
        story_so_far: The story text to continue
        temperature: Controls randomness
        max_tokens: Maximum length of the generated continuation
    
    Returns:
        The complete generated story continuation
    """
    prompt = f"""You are a creative storyteller. Continue the following story in an imaginative and engaging way.
Maintain the same style and tone as the original story. Write approximately 2-3 sentences.

Story so far:
{story_so_far}

Continue the story:"""
    
    payload = {
        "model": "local-model",
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "temperature": temperature,
        "max_tokens": max_tokens,
        "stream": True  # Enable streaming
    }
    
    headers = {
        "Content-Type": "application/json"
    }
    
    full_response = ""
    
    try:
        # Make request with stream=True to enable streaming
        response = requests.post(
            LM_STUDIO_URL,
            json=payload,
            headers=headers,
            stream=True,  # Important: enable streaming in requests
            timeout=30
        )
        
        if response.status_code == 200:
            print("Streaming response (real-time):")
            print("-" * 70)
            
            # Process each line in the stream
            for line in response.iter_lines():
                if line:
                    # Decode bytes to string
                    line_text = line.decode('utf-8')
                    
                    # SSE format: lines start with "data: "
                    if line_text.startswith('data: '):
                        # Extract JSON data
                        json_str = line_text[6:]  # Remove "data: " prefix
                        
                        # Check for [DONE] marker
                        if json_str.strip() == '[DONE]':
                            print("\n" + "-" * 70)
                            print("Stream complete!")
                            break
                        
                        try:
                            # Parse JSON chunk
                            chunk = json.loads(json_str)
                            
                            # Extract content delta
                            if 'choices' in chunk and len(chunk['choices']) > 0:
                                delta = chunk['choices'][0].get('delta', {})
                                content = delta.get('content', '')
                                
                                if content:
                                    # Print immediately (real-time effect)
                                    print(content, end='', flush=True)
                                    full_response += content
                                    
                        except json.JSONDecodeError:
                            # Skip invalid JSON lines
                            continue
            
            print(f"\n\nComplete response length: {len(full_response)} characters")
            return full_response
        else:
            print(f"Error: API returned status code {response.status_code}")
            print(f"Response: {response.text}")
            return ""
            
    except Exception as e:
        print(f"Error generating streaming story continuation: {str(e)}")
        return ""

# Example: Compare streaming vs non-streaming
print("=" * 70)
print("Example: Streaming Story Generation")
print("=" * 70)
test_story = "In a magical forest, there lived a glowing rabbit..."
print(f"\nOriginal story: {test_story}\n")
print("Generating continuation with streaming (watch it appear in real-time):\n")

streaming_result = generate_story_streaming(test_story, max_tokens=150)


### Key Differences: Streaming vs Non-Streaming

| Feature | Non-Streaming | Streaming |
|---------|---------------|-----------|
| **Request** | `"stream": false` or omitted | `"stream": true` |
| **Response** | Single JSON object | Multiple SSE chunks |
| **Processing** | Wait for complete response | Process chunks as they arrive |
| **User Experience** | Wait, then see all text | See text appear in real-time |
| **Code Complexity** | Simpler | More complex (need to parse chunks) |
| **Use Case** | When you need complete response | When you want real-time feedback |

### When to Use Streaming?

✅ **Use Streaming When**:
- Building interactive chat interfaces
- Users need immediate feedback
- Generating long responses
- Creating real-time applications

❌ **Avoid Streaming When**:
- You need the complete response before processing
- Simple one-off requests
- Error handling is complex
- You're processing the response as a whole

### Advanced: Handling Streaming Errors

When streaming, errors can occur mid-stream. Always check for error chunks:

```python
if 'error' in chunk:
    print(f"Error: {chunk['error']}")
    break
```


In [ ]:
def test_connection() -> bool:
    """
    Test if LM Studio is running and accessible.
    
    Returns:
        bool: True if connection successful, False otherwise
    """
    try:
        # Simple test message
        payload = {
            "model": "local-model",  # LM Studio uses "local-model" as the model name
            "messages": [
                {"role": "user", "content": "Hello! Please respond with 'Connection successful!'"}
            ],
            "temperature": 0.7,
            "max_tokens": 50
        }
        
        headers = {
            "Content-Type": "application/json"
        }
        
        response = requests.post(LM_STUDIO_URL, json=payload, headers=headers, timeout=10)
        
        if response.status_code == 200:
            result = response.json()
            assistant_message = result["choices"][0]["message"]["content"]
            print("✅ Connection successful!")
            print(f"Response: {assistant_message}")
            return True
        else:
            print(f"❌ Connection failed with status code: {response.status_code}")
            print(f"Response: {response.text}")
            return False
            
    except requests.exceptions.ConnectionError:
        print("❌ Connection error: Could not reach LM Studio.")
        print("   Please ensure LM Studio is running and the server is started on port 1234.")
        return False
    except Exception as e:
        print(f"❌ Error: {str(e)}")
        return False

# Test the connection
test_connection()


## Step 4: Understanding LLM and Prompt Engineering

### What is an LLM?

Large Language Models (LLMs) are AI systems trained on vast amounts of text data. They can:
- Generate human-like text
- Understand context and follow instructions
- Continue or complete text based on prompts

### Prompt Engineering Basics

**Prompt engineering** is the art of crafting effective instructions for LLMs. Key principles:

1. **Be Clear and Specific**: Clearly state what you want
2. **Provide Context**: Give relevant background information
3. **Use Examples**: Show the desired format or style
4. **Set Constraints**: Define length, style, or other requirements

### Example Prompt Structure for Story Continuation

```
You are a creative storyteller. Continue the following story in an imaginative way.
Make it engaging and maintain the same style and tone.

Story so far: [previous story text]

Continue the story:
```


## Step 5: Create a Function to Generate Story Continuations

Now we'll create a reusable function that sends a prompt to LM Studio and returns the story continuation. This function will handle the API communication and error handling.


In [ ]:
def generate_story_continuation(
    story_so_far: str,
    temperature: float = 0.8,
    max_tokens: int = 200
) -> Optional[str]:
    """
    Generate a story continuation using LM Studio.
    
    Args:
        story_so_far: The story text to continue
        temperature: Controls randomness (0.0 = deterministic, 1.0 = creative)
        max_tokens: Maximum length of the generated continuation
    
    Returns:
        The generated story continuation, or None if an error occurred
    """
    # Craft the prompt using prompt engineering principles
    prompt = f"""You are a creative storyteller. Continue the following story in an imaginative and engaging way.
Maintain the same style and tone as the original story. Write approximately 2-3 sentences.

Story so far:
{story_so_far}

Continue the story:"""
    
    payload = {
        "model": "local-model",
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "temperature": temperature,
        "max_tokens": max_tokens
    }
    
    headers = {
        "Content-Type": "application/json"
    }
    
    try:
        response = requests.post(LM_STUDIO_URL, json=payload, headers=headers, timeout=30)
        
        if response.status_code == 200:
            result = response.json()
            continuation = result["choices"][0]["message"]["content"].strip()
            return continuation
        else:
            print(f"Error: API returned status code {response.status_code}")
            print(f"Response: {response.text}")
            return None
            
    except Exception as e:
        print(f"Error generating story continuation: {str(e)}")
        return None

# Test the function with a simple story beginning
test_story = "In a magical forest, there lived a glowing rabbit..."
print("Original story beginning:")
print(test_story)
print("\nGenerating continuation...")
continuation = generate_story_continuation(test_story)
if continuation:
    print("\nGenerated continuation:")
    print(continuation)


## Step 6: Advanced Prompt Engineering Techniques

Let's explore different prompt engineering techniques to improve our story generation. We'll create variations that demonstrate different storytelling styles and approaches.


In [ ]:
def generate_story_with_style(
    story_so_far: str,
    style: str = "creative",
    temperature: float = 0.8,
    max_tokens: int = 200
) -> Optional[str]:
    """
    Generate story continuation with different styles using advanced prompt engineering.
    
    Args:
        story_so_far: The story text to continue
        style: Story style - "creative", "mysterious", "humorous", "dramatic"
        temperature: Controls randomness
        max_tokens: Maximum length of the generated continuation
    
    Returns:
        The generated story continuation, or None if an error occurred
    """
    # Different prompt templates for different styles
    style_prompts = {
        "creative": """You are a creative storyteller. Continue the following story with vivid imagery and imaginative details.
Make it engaging and maintain the same style and tone.

Story so far:
{story}

Continue the story with creative flair:""",
        
        "mysterious": """You are a mystery writer. Continue the following story with an air of mystery and suspense.
Add intriguing elements that make readers want to know more.

Story so far:
{story}

Continue the story with mystery:""",
        
        "humorous": """You are a humorous storyteller. Continue the following story with wit and humor.
Make it entertaining and light-hearted.

Story so far:
{story}

Continue the story with humor:""",
        
        "dramatic": """You are a dramatic storyteller. Continue the following story with emotional depth and intensity.
Create tension and emotional impact.

Story so far:
{story}

Continue the story with drama:"""
    }
    
    prompt_template = style_prompts.get(style, style_prompts["creative"])
    prompt = prompt_template.format(story=story_so_far)
    
    payload = {
        "model": "local-model",
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "temperature": temperature,
        "max_tokens": max_tokens,
        "stream": True  # Enable streaming
    }
    
    headers = {
        "Content-Type": "application/json"
    }
    
    try:
        # Make request with stream=True to enable SSE streaming
        response = requests.post(
            LM_STUDIO_URL,
            json=payload,
            headers=headers,
            stream=True,  # Enable streaming in requests
            timeout=30
        )
        
        if response.status_code == 200:
            continuation = process_streaming_response(response, show_streaming=True)
            return continuation if continuation else None
        else:
            print(f"Error: API returned status code {response.status_code}")
            return None
            
    except Exception as e:
        print(f"Error generating story continuation: {str(e)}")
        return None

# Example: Try different styles
example_story = "The old clock in the tower struck midnight, and something strange began to happen..."
print("Original story:")
print(example_story)
print("\n" + "="*60)

for style in ["creative", "mysterious", "humorous", "dramatic"]:
    print(f"\n{style.upper()} style continuation:")
    continuation = generate_story_with_style(example_story, style=style)
    # Note: The continuation is already displayed during streaming, so we don't print it again
    if not continuation:
        print("Failed to generate continuation.")
    print("-" * 60)


## Step 7: Interactive Story Relay System

Now let's create a complete interactive story relay system. This will allow you to start a story and have the AI continue it, creating a collaborative storytelling experience.


In [ ]:
class StoryRelayRobot:
    """
    A class to manage interactive story relay sessions.
    This demonstrates object-oriented programming with API integration.
    """
    
    def __init__(self, api_url: str = LM_STUDIO_URL, temperature: float = 0.8):
        """
        Initialize the Story Relay Robot.
        
        Args:
            api_url: The LM Studio API endpoint URL
            temperature: Default temperature for story generation
        """
        self.api_url = api_url
        self.temperature = temperature
        self.story_history: List[str] = []
        self.full_story: str = ""
    
    def start_story(self, beginning: str) -> str:
        """
        Start a new story with the given beginning.
        
        Args:
            beginning: The story's opening sentence(s)
        
        Returns:
            The beginning text
        """
        self.story_history = [beginning]
        self.full_story = beginning
        return beginning
    
    def continue_story(self, max_tokens: int = 200) -> Optional[str]:
        """
        Continue the current story using AI generation.
        
        Args:
            max_tokens: Maximum length of the continuation
        
        Returns:
            The generated continuation, or None if an error occurred
        """
        if not self.story_history:
            print("No story started yet. Please use start_story() first.")
            return None
        
        # Get the current story
        current_story = self.full_story
        
        # Generate continuation with streaming
        print(f"[Generating continuation {len(self.story_history)}...]")
        continuation = generate_story_continuation(
            current_story,
            temperature=self.temperature,
            max_tokens=max_tokens
        )
        
        if continuation:
            self.story_history.append(continuation)
            self.full_story += " " + continuation
            return continuation
        else:
            return None
    
    def get_full_story(self) -> str:
        """Get the complete story so far."""
        return self.full_story
    
    def get_story_history(self) -> List[str]:
        """Get the story as a list of segments."""
        return self.story_history.copy()
    
    def reset(self):
        """Reset the story to start fresh."""
        self.story_history = []
        self.full_story = ""

# Create an instance of the Story Relay Robot
robot = StoryRelayRobot(temperature=0.1)
print("Story Relay Robot initialized!")
print("Use robot.start_story('your beginning') to start a new story")
print("Use robot.continue_story() to generate continuations")


## Step 8: Example Story Relay Session

Let's demonstrate a complete story relay session. This shows how the system works in practice.


In [ ]:
# Start a new story
story_beginning = "In a magical forest, there lived a glowing rabbit named Luna who could speak to the stars."
print("📖 Starting a new story...")
print(f"\nBeginning: {story_beginning}\n")

robot.start_story(story_beginning)

# Generate multiple continuations to build the story
print("=" * 70)
print("STORY RELAY SESSION")
print("=" * 70)

for i in range(3):  # Generate 3 continuations
    print(f"\n[Turn {i+1}] Generating continuation...")
    continuation = robot.continue_story(max_tokens=150)
    
    if continuation:
        print(f"\n✓ Continuation {i+1} complete!")
    else:
        print("Failed to generate continuation.")
        break
    
    print("-" * 70)

# Display the complete story
print("\n" + "=" * 70)
print("COMPLETE STORY")
print("=" * 70)
print(robot.get_full_story())
print("=" * 70)


## Step 9: Experiment with Your Own Stories

Now it's your turn! Try creating your own stories. You can:

1. **Start your own story**: Use `robot.start_story("Your story beginning here")`
2. **Continue the story**: Use `robot.continue_story()` to generate AI continuations
3. **Try different styles**: Use `generate_story_with_style()` with different style options
4. **Adjust parameters**: Experiment with different `temperature` values (0.0-1.0) and `max_tokens`

### Tips for Better Results:

- **Temperature**: 
  - Lower (0.3-0.5): More focused and consistent
  - Higher (0.7-0.9): More creative and varied
  
- **Max Tokens**: 
  - 100-150: Short continuations
  - 200-300: Medium-length continuations
  - 400+: Longer, more detailed continuations

- **Prompt Quality**: 
  - Provide clear context
  - Specify desired style or tone
  - Give examples if needed


In [ ]:
# Example: Create your own story
# Uncomment and modify the code below to create your own story

my_story_beginning = "Once upon a time, in a world where technology and magic coexisted..."
robot.start_story(my_story_beginning)
print(f"Your story: {my_story_beginning}\n")

# Generate continuations
for i in range(2):
    continuation = robot.continue_story()
    if continuation:
        print(f"Continuation {i+1}: {continuation}\n")

print("Full story:")
print(robot.get_full_story())


## Summary and Key Learnings

Congratulations! You've learned:

### 1. RESTful API Concepts
- How to make HTTP POST requests using the `requests` library
- Understanding API endpoints, payloads, and responses
- Working with JSON data format

### 2. LLM Fundamentals
- How Large Language Models generate text
- The role of prompts in guiding AI responses
- Understanding temperature and token limits

### 3. Prompt Engineering
- Crafting clear and effective prompts
- Using different styles and tones
- Providing context and constraints

### 4. Practical Implementation
- Creating reusable functions for API interactions
- Building an object-oriented story relay system
- Error handling and connection testing

### Next Steps

- Experiment with different prompt templates
- Try combining multiple story styles
- Create a more advanced interface (e.g., using Streamlit or Gradio)
- Explore other LM Studio features and model parameters
- Learn about system prompts and few-shot learning

Happy storytelling! 🎭📚
